# Standalone intent training on Google Colab

Run cells in order. Open this notebook on Colab and upload the generated
`colab-upload.zip`; no Git clone or project source is needed. Select a GPU
runtime and provision an unchanged local E5 model folder before preflight.
E5 embeds on GPU; sklearn heads train on CPU. This notebook only calls the
existing standalone CLI. It does not train automatically when opened.

The bundled 600-case sample is **not human-reviewed**. Training must stop until
you supply a genuinely reviewed private dataset with truthful review notes.
No cell marks it reviewed. Agent labels, filenames and checkboxes are not review.
Only upload cloud-approved content, never `.env` or employee/database credentials.

Training produces a candidate, not an accepted runtime model. A separate internal
benchmark remains mandatory. No automatic retries, model downloads or Drive mount.
Local mocked checks are not evidence that live Colab, CUDA or training succeeded.

## 1. Configure paths

Use Colab Files to upload the kit ZIP to `/content/colab-upload.zip`. Put the
exact hydrated E5 model folder at `MODEL_PATH` yourself. Model revision must match
the internal runtime. Do not put reports or notebooks inside the model folder.
Keep `WORKSPACE=/content`; all working data and exports go under its `.tmp`.

`UPLOAD_REVIEWED_DATASET=True` enables the later explicit upload/replacement cell.
Leave it false to retain an existing working dataset. No setting bypasses review.
Rerunning this cell clears the selected run; it does not erase saved files.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json
import os
import re
import stat
import subprocess
import sys
import uuid
import zipfile

WORKSPACE = Path('/content')
KIT_ZIP = WORKSPACE / 'colab-upload.zip'
KIT_DIR = WORKSPACE / 'intent-colab-kit'
MODEL_PATH = WORKSPACE / 'e5-model'
MODEL_REVISION = '614241f622f53c4eeff9890bdc4f31cfecc418b3'
DEVICE = 'cuda'
BATCH_SIZE = 16
UPLOAD_REVIEWED_DATASET = False

WORK_DIR = WORKSPACE / '.tmp/intent-colab'
DATASET_PATH = WORK_DIR / 'reviewed-cases.json'
RUN_ROOT = WORK_DIR / 'runs'
LAST_RUN = None


## Workflow helpers

These helpers handle files, CLI execution and downloads; the model/training
algorithm stays in `train_intent_standalone.py`. ZIP checks establish integrity
against its included manifest, not publisher authenticity. Use a trusted kit.
Only safe status fields are displayed, never dataset questions or embeddings.

In [ ]:
def need(condition, message):
    if not condition:
        raise RuntimeError(message)


def no_links(path):
    path = Path(os.path.abspath(path))
    for part in (*reversed(path.parents), path):
        if part.is_symlink():
            raise RuntimeError('Linked paths are not supported.')
        if part.exists():
            need(not (getattr(part.lstat(), 'st_file_attributes', 0) & 0x400),
                 'Linked paths are not supported.')


def private_path(path):
    no_links(path)
    path = Path(path).resolve()
    root = (WORKSPACE / '.tmp').resolve()
    need(path != root and path.is_relative_to(root), 'Working files must stay under workspace .tmp.')
    return path


def digest_json(value):
    serialized = json.dumps(value, sort_keys=True, ensure_ascii=False,
                            allow_nan=False, separators=(',', ':'))
    return hashlib.sha256(serialized.encode('utf-8')).hexdigest()


def colab_files():
    from google.colab import files
    return files


def prepare_kit():
    no_links(KIT_ZIP)
    need(KIT_ZIP.is_file(), 'Upload the trusted colab-upload.zip with Colab Files first.')
    required = {'train_intent_standalone.py', 'requirements-colab.txt', 'README.md',
                'draft-cases.json', 'dataset-review.json', 'notebooks/train_intent.ipynb'}
    with zipfile.ZipFile(KIT_ZIP) as archive:
        infos = archive.infolist()
        names = [info.filename for info in infos]
        need(len(names) == len(set(names)), 'Duplicate archive entries.')
        need(set(names) in (required | {'bundle-manifest.json'},
                            required | {'bundle-manifest.json', 'verification.json'}),
             'Unexpected or missing archive entries; rebuild the current kit.')
        need(sum(info.file_size for info in infos) <= 10 * 1024 * 1024, 'Kit archive is too large.')
        for info in infos:
            name = PurePosixPath(info.filename)
            need(not name.is_absolute() and '..' not in name.parts
                 and '\\' not in info.filename and not info.is_dir()
                 and not stat.S_ISLNK(info.external_attr >> 16), 'Unsafe archive entry.')
        payloads = {info.filename: archive.read(info) for info in infos}
    manifest = json.loads(payloads['bundle-manifest.json'])
    hashes = manifest.get('files_sha256', {})
    need(isinstance(hashes, dict) and set(hashes) == set(payloads) - {'bundle-manifest.json'},
         'Incomplete archive manifest.')
    for name, expected in hashes.items():
        need(hashlib.sha256(payloads[name]).hexdigest() == expected, 'Kit checksum mismatch.')
    no_links(KIT_DIR)
    if KIT_DIR.exists():
        need(KIT_DIR.is_dir(), 'Kit destination is not a directory.')
        for name, content in payloads.items():
            target = KIT_DIR / name
            no_links(target)
            need(target.is_file() and target.read_bytes() == content,
                 'Existing kit differs. Choose a fresh KIT_DIR; files were not overwritten.')
        print('Verified existing kit; files retained.')
        return
    staging = private_path(WORK_DIR / 'kit-staging' / uuid.uuid4().hex)
    staging.mkdir(parents=True, exist_ok=False)
    for name, content in payloads.items():
        target = staging / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
    KIT_DIR.parent.mkdir(parents=True, exist_ok=True)
    staging.rename(KIT_DIR)
    print('Kit extracted and checksums verified.')


def install_dependencies():
    requirement = KIT_DIR / 'requirements-colab.txt'
    need(requirement.is_file(), 'Prepare and verify the kit first.')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(requirement)], check=True)
    print('Installation finished. Restart the runtime if pip requests it, then rerun setup cells.')


def check_environment():
    no_links(MODEL_PATH)
    need(MODEL_PATH.is_dir(), 'Provision the unchanged local E5 folder at MODEL_PATH.')
    need(re.fullmatch('[0-9a-fA-F]{40}', MODEL_REVISION) is not None, 'Expected a 40-hex model revision.')
    need(DEVICE in ('cuda', 'cpu'), 'Select cuda or cpu explicitly.')
    need(type(BATCH_SIZE) is int and BATCH_SIZE > 0, 'Batch size must be a positive integer.')
    if DEVICE == 'cuda':
        import torch
        need(torch.cuda.is_available(), 'CUDA unavailable. Select a GPU runtime or explicitly choose cpu.')
    print('Model path and selected device checked; model compatibility is checked by preflight.')


def prepare_dataset():
    target = private_path(DATASET_PATH)
    if target.exists():
        need(target.is_file(), 'Working dataset path is not a regular file.')
        print('Existing working dataset retained; no review edits were overwritten.')
        return
    source = KIT_DIR / 'draft-cases.json'
    need(source.is_file(), 'Prepare the kit first.')
    target.parent.mkdir(parents=True, exist_ok=True)
    with target.open('xb') as output:
        output.write(source.read_bytes())
    print('Unreviewed sample copied to private workspace. Human review is still required.')


def run_cli(command, extra=(), allowed=(0,), dataset_path=None):
    need(command in ('validate', 'preflight', 'train'), 'Unsupported workflow step.')
    trainer = KIT_DIR / 'train_intent_standalone.py'
    need(trainer.is_file(), 'Prepare the kit first.')
    dataset = private_path(DATASET_PATH if dataset_path is None else dataset_path)
    result = subprocess.run([sys.executable, str(trainer), command, '--dataset', str(dataset), *extra],
                            cwd=str(WORKSPACE), stdout=subprocess.PIPE, text=True, check=False)
    try:
        payload = json.loads(result.stdout)
    except (ValueError, TypeError):
        raise RuntimeError('CLI did not return one valid JSON result; do not continue.') from None
    need(isinstance(payload, dict), 'Invalid CLI result.')
    safe = {k: payload[k] for k in ('status', 'error_code', 'run_directory') if k in payload}
    validation = payload.get('validation', {})
    if isinstance(validation, dict):
        safe.update({k: validation[k] for k in ('case_count', 'review_required', 'training_allowed')
                     if k in validation})
        safe['structural_error_count'] = len(validation.get('errors', []))
        safe['structural_errors_first_10'] = [
            {k: row[k] for k in ('code', 'case_index', 'field', 'split') if k in row}
            for row in validation.get('errors', [])[:10]]
    print(json.dumps(safe))
    need(result.returncode in allowed, 'CLI step failed. Resolve the reported status before continuing.')
    return payload


def upload_reviewed_dataset():
    uploaded = colab_files().upload()
    need(len(uploaded) == 1, 'Upload exactly one reviewed dataset JSON.')
    name, content = next(iter(uploaded.items()))
    need(name.lower().endswith('.json') and len(content) <= 10 * 1024 * 1024,
         'Expected one dataset JSON, at most 10 MiB.')
    try:
        json.loads(content)
    except (ValueError, UnicodeError):
        raise RuntimeError('Uploaded dataset is not valid JSON; existing copy retained.') from None
    target = private_path(DATASET_PATH)
    target.parent.mkdir(parents=True, exist_ok=True)
    staged = private_path(target.parent / ('upload-' + uuid.uuid4().hex + '.json'))
    try:
        with staged.open('xb') as output:
            output.write(content)
        result = run_cli('validate', dataset_path=staged)
        need(result.get('validation', {}).get('training_allowed') is True,
             'Dataset is not reviewed/valid; existing copy retained.')
        os.replace(staged, target)
    finally:
        staged.unlink(missing_ok=True)
    print('Explicit upload replaced the private working dataset; the kit sample was not modified.')


def model_arguments():
    return ['--model-path', str(MODEL_PATH), '--model-revision', MODEL_REVISION,
            '--device', DEVICE, '--batch-size', str(BATCH_SIZE)]


def selected_run(path):
    need(path is not None, 'No completed run selected. Run the train cell first.')
    path = private_path(path)
    root = private_path(RUN_ROOT)
    need(path != root and path.is_relative_to(root), 'Run must belong to the configured run directory.')
    return path


def completed_manifest(path):
    manifest_path = path / 'manifest.json'
    no_links(manifest_path)
    need(manifest_path.is_file(), 'Run manifest is missing.')
    manifest = json.loads(manifest_path.read_bytes())
    need(manifest.get('complete') is True
         and manifest.get('status') in ('candidate_ready_for_benchmark', 'quality_failed'),
         'Run is incomplete or failed; it is not exportable as a completed batch.')
    return manifest


def train_once():
    global LAST_RUN
    LAST_RUN = None
    validation = run_cli('validate')
    need(validation.get('validation', {}).get('training_allowed') is True,
         'Human review required: inspect labels/splits and provide truthful review notes before training.')
    check_environment()
    result = run_cli('train', [*model_arguments(), '--output-root', str(private_path(RUN_ROOT))],
                     allowed=(0, 1))
    path = selected_run(result.get('run_directory'))
    manifest = completed_manifest(path)
    need(manifest['status'] == result.get('status'), 'Run status and manifest disagree.')
    need(digest_json(json.loads(private_path(DATASET_PATH).read_bytes())) == manifest.get('dataset_sha256'),
         'Working dataset changed; retain the exact training dataset before exporting.')
    LAST_RUN = path
    print('Completed candidate batch selected; benchmark still required.'
          if manifest['status'] == 'candidate_ready_for_benchmark'
          else 'Quality failed: selected for diagnostics only, not runtime or promotion.')
    return path


def export_run(path):
    path = selected_run(path)
    manifest = completed_manifest(path)
    dataset = private_path(DATASET_PATH).read_bytes()
    need(digest_json(json.loads(dataset)) == manifest.get('dataset_sha256'),
         'Dataset hash mismatch; export stopped to avoid pairing the wrong dataset.')
    payloads = {'reviewed-cases.json': dataset}
    allowed = ('validation.json', 'preflight.json', 'training-report.json', 'candidate.json', 'manifest.json')
    for name in allowed:
        item = path / name
        no_links(item)
        need(item.is_file(), 'Completed run is missing a required output.')
        content = item.read_bytes()
        if name == 'manifest.json':
            need(digest_json(json.loads(content)) == digest_json(manifest),
                 'Manifest changed during export; export stopped.')
        if name in ('candidate.json', 'training-report.json'):
            need(digest_json(json.loads(content)) == manifest.get('outputs', {}).get(name),
                 'Run output hash mismatch; export stopped.')
        payloads['runs/' + path.name + '/' + name] = content
    output_dir = private_path(WORK_DIR / 'exports')
    output_dir.mkdir(parents=True, exist_ok=True)
    archive_path = output_dir / (path.name + '-' + uuid.uuid4().hex + '.zip')
    with zipfile.ZipFile(archive_path, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
        for name, content in payloads.items():
            archive.writestr(name, content)
    with zipfile.ZipFile(archive_path) as archive:
        need(archive.testzip() is None, 'Export archive integrity failure.')
    print('Export contains the selected run and exact dataset only. It is not acceptance evidence.')
    return archive_path


## 2. Verify and extract the uploaded kit

Upload `colab-upload.zip` through Colab Files before running this cell. An existing
kit is reused only when all packaged files match. A differing kit is never
overwritten; choose a fresh `KIT_DIR` or deliberately resolve the difference.

In [ ]:
prepare_kit()


## 3. Install dependencies

This cell installs the kit's requirements into this Colab runtime. Resolve
incompatibilities normally; never use force or legacy dependency resolution.
If a runtime restart is needed, rerun setup cells afterward, then continue here.

In [ ]:
install_dependencies()


## 4. Check the provisioned model path and selected device

The model is supplied separately. A missing folder or unavailable CUDA stops the
step. Selecting a GPU does not move the sklearn heads to GPU. Explicit CPU runs
are supported by setting `DEVICE="cpu"` in configuration.

In [ ]:
check_environment()


## 5. Prepare a private dataset

The kit sample stays unmodified. Rerunning retains the existing working JSON.
Human review must check every source/style label, critical case, scope and
semantic/template split independence. Read `dataset-review.json` for limitations.
Review notes in that file attest only the original agent-audited draft.

In [ ]:
prepare_dataset()


## 6. Validate and inspect review status

For the shipped draft, expect `review_required=true`, `training_allowed=false`.
Structural success does not certify labels. After actual human review, upload
your privately reviewed JSON in the optional next cell, or edit only the private
working copy through Colab Files with truthful `review_notes` and `reviewed=true`.
No cell changes that flag. Do not tune labels or training using holdout predictions.

In [ ]:
VALIDATION = run_cli('validate')


### Optional: explicitly replace the private working dataset

Set `UPLOAD_REVIEWED_DATASET=True` in configuration to use this cell. It prompts
for one JSON, validates its declared review/structure before replacing the working
copy, and preserves the existing copy on failure. A boolean declaration does not
prove human review; supply only data you actually reviewed. Re-run validation.

In [ ]:
if UPLOAD_REVIEWED_DATASET:
    upload_reviewed_dataset()
else:
    print('Upload disabled; existing working dataset retained.')


## 7. Preflight

Checks all token lengths and model identity. CUDA additionally compares CPU/GPU
embeddings of three fixed fictional questions; this is diagnostic, not acceptance.
Draft preflight is allowed. No dataset embeddings are generated in this step.

In [ ]:
check_environment()
PREFLIGHT = run_cli('preflight', model_arguments())


## 8. Train one new batch

This cell validates again and refuses unreviewed data before starting training.
It creates a new run each time; there is no resume, automatic retry or persistent
embedding cache. Lower `BATCH_SIZE` deliberately after OOM and start a fresh run.
Only completed manifests are selected. Quality failure can be exported for
diagnostics but cannot authorize benchmark promotion or runtime use.

In [ ]:
train_once()


## 9. Export and download the selected batch

The archive contains exactly the selected run's five files and the exact dataset
whose canonical hash matches its manifest. It excludes all other runs, models,
kit files, caches and credentials. Changed datasets or outputs stop export.
The active run selection is lost on runtime restart; saved files are not erased.

In [ ]:
EXPORT_PATH = export_run(LAST_RUN)
colab_files().download(str(EXPORT_PATH))


## 10. Benchmark internally before enabling intent

Extract the downloaded results to a fresh directory under the internal repository
`.tmp`. Use the existing `scripts.benchmark_intent` command with the exact dataset,
candidate, training report, matching E5 model/revision, provisioned corpus/database,
reranker and independent matching relevance profile; use at least 200 paired
requests. Follow the kit README for arguments.

Only a passing internal benchmark emits `*-accepted.json`. Never rename a candidate
to bypass acceptance. Provision accepted output explicitly, start with observe and
restart only in an authorized rollout. Assist requires separate semantic evaluation.
No cell connects to the database, changes `.env` or enables runtime intent.